# CreaseIQ — Agent Playground (Milestone 1)

Scratch notebook for learning/validating the LangGraph foundation with **mock cricket data** only (no real rules, no RAG yet).

Sections:
1. Setup & Sarvam connection
2. Mock retrieval tool
3. Exercise A — prebuilt `create_agent`
4. Exercise B — custom `StateGraph` (understand → check context → clarify / retrieve → answer)
5. Multi-turn clarification with a checkpointer
6. Languages: English (default) and Assamese, with an optional response-language override

Create `backend/.env` from `.env.example` and set `SARVAM_API_KEY` before running the LLM cells.

## 1. Setup

In [ ]:
import json, os, re
from pathlib import Path
from typing import Literal, Optional, TypedDict, Annotated

from dotenv import load_dotenv
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

# Look for .env in backend/ (parent of src/) or the current directory
load_dotenv(Path.cwd().parent / ".env")
load_dotenv()

SARVAM_BASE_URL = os.getenv("SARVAM_BASE_URL", "https://api.sarvam.ai/v2")
SARVAM_MODEL = os.getenv("SARVAM_MODEL", "sarvam-105b")
print("API key set:", bool(os.getenv("SARVAM_API_KEY")), "| model:", SARVAM_MODEL)
print("LangSmith tracing:", os.getenv("LANGSMITH_TRACING", "false"), "| project:", os.getenv("LANGSMITH_PROJECT", "default"),
      "| key set:", bool(os.getenv("LANGSMITH_API_KEY")))

### 1a. Model connection

Per the spec, verify Sarvam's **basic chat** first, and **tool calling separately** (don't assume one implies the other).
Auth header format must be checked against current Sarvam docs — the default below sends the key as a Bearer token via `api_key`; if Sarvam requires an `api-subscription-key` header, uncomment `default_headers`.

In [ ]:
llm = ChatOpenAI(
    model=SARVAM_MODEL,
    base_url=SARVAM_BASE_URL,
    api_key=os.getenv("SARVAM_API_KEY", "missing"),
    # default_headers={"api-subscription-key": os.getenv("SARVAM_API_KEY", "")},
    temperature=0,
    timeout=60,
    max_retries=1,
)

In [ ]:
# Test 1: basic text completion
print(llm.invoke("In one sentence, what is a no-ball in cricket?").content)

In [ ]:

# Test 2: Assamese - does the model reply in Assamese script, and render cricket terms sensibly?
print(llm.invoke("ক্ৰিকেটত নো-বল মানে কি? অসমীয়াত এটা বাক্যত উত্তৰ দিয়ক।").content)


## 2. Mock retrieval tool
Stand-in for `search_cricket_rules`. Returns canned, **fake** passages — not real MCC/ICC text.

In [ ]:
MOCK_RULES = [
    {"source": "MOCK Laws of Cricket", "law": "21", "title": "No ball",
     "text": "MOCK: A delivery is a no ball if the bowler's front foot lands beyond the popping crease."},
    {"source": "MOCK Laws of Cricket", "law": "30", "title": "Bowled",
     "text": "MOCK: The striker is out bowled if the wicket is put down by a ball delivered by the bowler."},
    {"source": "MOCK ICC T20I Playing Conditions", "law": "PC-4", "title": "Fielding restrictions",
     "text": "MOCK: During the powerplay a maximum of two fielders are permitted outside the inner circle."},
]

@tool
def search_cricket_rules(query: str, format: Optional[str] = None) -> str:
    """Search the cricket rules knowledge base. `format` is 'odi' or 't20i'."""
    words = set(re.findall(r"\w+", query.lower()))
    hits = [r for r in MOCK_RULES if words & set(re.findall(r"\w+", (r["title"] + " " + r["text"]).lower()))]
    return json.dumps(hits[:3]) if hits else "NO_RESULTS"

print(search_cricket_rules.invoke({"query": "no ball front foot"}))

## 3. Exercise A — prebuilt `create_agent`
Learning only. Production uses the custom graph in section 4. This also tests Sarvam **tool calling** and multi-turn tool-call messages.

In [ ]:
from langchain.agents import create_agent

simple_agent = create_agent(
    llm,
    tools=[search_cricket_rules],
    system_prompt="You are a cricket rules assistant. Use the tool, answer briefly, cite the law number. Never invent rules.",
)

result = simple_agent.invoke({"messages": [HumanMessage("Is it a no ball if the front foot is over the line?")]})
for m in result["messages"]:
    m.pretty_print()

## 4. Exercise B — custom `StateGraph`

```
START → understand_question → check_context ─┬─ unsupported → handle_unsupported → END
                                              ├─ missing facts → ask_clarification → END
                                              └─ ok → retrieve_rules → generate_answer → END
```
Nodes are small functions over a typed state. The LLM is asked for JSON in `understand_question` (plain-text JSON, so this doesn't depend on structured-output/tool support).

In [ ]:
class CreaseState(TypedDict, total=False):
    messages: Annotated[list, add_messages]
    question: str
    requested_language: Optional[str]  # optional caller preference (e.g. from the API/UI); None = follow the user
    input_language: str              # detected: en | as | other
    response_language: str           # language the answer is written in: en | as
    search_query_en: str             # English retrieval query (rules sources are English)
    intent: str                      # rule_lookup | scenario | comparison | follow_up | out_of_scope
    format: str                      # odi | t20i | unknown
    scenario_facts: list[str]
    missing_information: list[str]
    clarification_question: Optional[str]
    retrieved_rules: list[dict]
    answer: str
    status: str                      # answered | clarification_required | unsupported | insufficient_evidence

In [ ]:
# ---- Language support -------------------------------------------------------
# English is the default/fallback. Add a new language by adding one entry to each dict.
SUPPORTED_LANGUAGES = {"en": "English", "as": "Assamese (অসমীয়া)"}
DEFAULT_LANGUAGE = "en"

# Fixed (non-LLM) messages. NOTE: Assamese strings need review by a native speaker.
STRINGS = {
    "en": {
        "unsupported": "I can only help with men's ODI and T20I cricket rules.",
        "no_evidence": "I couldn't verify this from the available sources.",
    },
    "as": {
        "unsupported": "মই কেৱল পুৰুষ ODI আৰু T20I ক্ৰিকেটৰ নিয়মৰ বিষয়ে সহায় কৰিব পাৰোঁ।",
        "no_evidence": "উপলব্ধ উৎসসমূহৰ পৰা মই ইয়াক নিশ্চিত কৰিব নোৱাৰিলোঁ।",
    },
}

def lang_instruction(code: str) -> str:
    name = SUPPORTED_LANGUAGES.get(code, SUPPORTED_LANGUAGES[DEFAULT_LANGUAGE])
    return (f"Write your reply in {name}. Keep law numbers, clause IDs, source titles and quoted rule text exactly as given "
            f"(do not translate or transliterate them); translate only your own explanation. "
            f"Do not claim the reply is an official translation.")

def pick_response_language(requested: Optional[str], detected: str) -> str:
    """Caller preference wins; otherwise mirror the user's language; unsupported -> English."""
    for code in (requested, detected):
        if code in SUPPORTED_LANGUAGES:
            return code
    return DEFAULT_LANGUAGE

# ---- Nodes ------------------------------------------------------------------
UNDERSTAND_PROMPT = """You analyse cricket-rules questions. The user may write in English or Assamese (Assamese script or romanised). Given the conversation, reply with ONLY a JSON object:
{
  "input_language": "en" | "as" | "other",
  "intent": "rule_lookup" | "scenario" | "comparison" | "follow_up" | "out_of_scope",
  "format": "odi" | "t20i" | "unknown",
  "scenario_facts": [facts the user explicitly stated, accumulated across the conversation, written in English],
  "missing_information": [facts that would materially change the answer and are NOT yet known, written in English],
  "search_query_en": "a short English query describing the rule being asked about"
}
Rules: input_language is the language of the LATEST user message; do not assume a format that was not stated; do not turn assumptions into facts; only list a missing fact if it could change the outcome.
Only men's ODI and T20I cricket are supported; anything else is out_of_scope."""

def parse_json(text: str) -> dict:
    m = re.search(r"\{.*\}", text, re.S)
    return json.loads(m.group(0)) if m else {}

def understand_question(state: CreaseState) -> CreaseState:
    resp = llm.invoke([SystemMessage(UNDERSTAND_PROMPT), *state["messages"]])
    try:
        data = parse_json(resp.content)
    except json.JSONDecodeError:
        data = {}
    question = state["messages"][-1].content
    detected = data.get("input_language", DEFAULT_LANGUAGE)
    return {
        "question": question,
        "input_language": detected,
        "response_language": pick_response_language(state.get("requested_language"), detected),
        "intent": data.get("intent", "rule_lookup"),
        "format": data.get("format", "unknown"),
        "scenario_facts": data.get("scenario_facts", []),
        "missing_information": data.get("missing_information", []),
        "search_query_en": data.get("search_query_en") or question,
    }

def check_context(state: CreaseState) -> CreaseState:
    # Routing happens in route_after_check; this node is a hook for extra deterministic checks.
    return {}

def ask_clarification(state: CreaseState) -> CreaseState:
    missing = state["missing_information"][0]  # one focused question at a time
    q = llm.invoke([
        SystemMessage("Ask ONE short, focused question to get this missing cricket fact. Do not repeat the scenario. "
                      + lang_instruction(state["response_language"])),
        HumanMessage(f"Scenario facts: {state.get('scenario_facts')}\nMissing: {missing}"),
    ]).content
    return {"clarification_question": q, "answer": q, "status": "clarification_required",
            "messages": [AIMessage(q)]}

def retrieve_rules(state: CreaseState) -> CreaseState:
    # Sources are English, so always search with the English query.
    raw = search_cricket_rules.invoke({"query": state["search_query_en"], "format": state.get("format")})
    return {"retrieved_rules": [] if raw == "NO_RESULTS" else json.loads(raw)}

def generate_answer(state: CreaseState) -> CreaseState:
    lang = state["response_language"]
    rules = state.get("retrieved_rules", [])
    if not rules:
        msg = STRINGS[lang]["no_evidence"]
        return {"answer": msg, "status": "insufficient_evidence", "messages": [AIMessage(msg)]}
    evidence = "\n".join(f"[{r['source']} | Law {r['law']}] {r['text']}" for r in rules)
    ans = llm.invoke([
        SystemMessage("Answer concisely using ONLY the evidence. State the outcome, one-line reason, then 'Rule reference: ...'. "
                      "Cite only laws present in the evidence. Note this is decision support, not an official umpire decision. "
                      + lang_instruction(lang)),
        HumanMessage(f"Facts: {state.get('scenario_facts')}\nQuestion: {state['question']}\nEvidence:\n{evidence}"),
    ]).content
    return {"answer": ans, "status": "answered", "messages": [AIMessage(ans)]}

def handle_unsupported(state: CreaseState) -> CreaseState:
    msg = STRINGS[state["response_language"]]["unsupported"]
    return {"answer": msg, "status": "unsupported", "messages": [AIMessage(msg)]}

In [ ]:
def route_after_check(state: CreaseState) -> Literal["handle_unsupported", "ask_clarification", "retrieve_rules"]:
    if state["intent"] == "out_of_scope":
        return "handle_unsupported"
    if state["missing_information"]:
        return "ask_clarification"
    return "retrieve_rules"

builder = StateGraph(CreaseState)
builder.add_node("understand_question", understand_question)
builder.add_node("check_context", check_context)
builder.add_node("ask_clarification", ask_clarification)
builder.add_node("retrieve_rules", retrieve_rules)
builder.add_node("generate_answer", generate_answer)
builder.add_node("handle_unsupported", handle_unsupported)

builder.add_edge(START, "understand_question")
builder.add_edge("understand_question", "check_context")
builder.add_conditional_edges("check_context", route_after_check)
builder.add_edge("retrieve_rules", "generate_answer")
for n in ("ask_clarification", "generate_answer", "handle_unsupported"):
    builder.add_edge(n, END)

graph = builder.compile(checkpointer=InMemorySaver())  # in-memory: local experiments only
graph

## 5. Run it (single turn + multi-turn clarification)
`thread_id` scopes the conversation state.

In [ ]:
def ask(text: str, thread_id: str = "demo-1", language: Optional[str] = None) -> CreaseState:
    """`language` = optional response-language override ('en' / 'as'); None = reply in the user's language."""
    out = graph.invoke(
        {"messages": [HumanMessage(text)], "requested_language": language},  # reset each turn
        config={"configurable": {"thread_id": thread_id}},
    )
    print(f"[{out['status']} | in={out['input_language']} out={out['response_language']}] {out['answer']}\n")
    return out

_ = ask("Is it a no ball if the bowler's front foot lands beyond the popping crease in a T20I?", "t1")

In [ ]:
# Clarification loop: first turn should ask, second turn (same thread) should answer
_ = ask("The batter left the crease and the ball hit the stumps. Is he out?", "t2")
_ = ask("It was an ODI and the ball was live.", "t2")

In [ ]:
# Out of scope
_ = ask("Who won the 2011 football world cup?", "t3")

### Assamese examples
The reply should follow the user's language. Pass `language=` to override (e.g. ask in Assamese, answer in English).

In [ ]:

# Assamese question -> Assamese answer (law numbers / source titles stay unchanged)
_ = ask("T20I ত বলাৰৰ সন্মুখৰ ভৰি পপিং ক্ৰীজৰ বাহিৰত পৰিলে সেয়া নো-বল নেকি?", "a1")


In [ ]:

# Assamese clarification loop (same thread)
_ = ask("বেটাৰে ক্ৰীজ এৰিলে আৰু বলটোৱে ষ্টাম্পত লাগিল। তেওঁ আউট নেকি?", "a2")
_ = ask("সেয়া ODI আছিল আৰু বলটো লাইভ আছিল।", "a2")


In [ ]:

# Assamese question, English answer (caller override)
_ = ask("T20I ত বলাৰৰ সন্মুখৰ ভৰি পপিং ক্ৰীজৰ বাহিৰত পৰিলে সেয়া নো-বল নেকি?", "a3", language="en")

# English question, Assamese answer
_ = ask("Is it a no ball if the front foot lands beyond the popping crease in a T20I?", "a4", language="as")

# Assamese out-of-scope -> fixed Assamese message
_ = ask("২০১১ চনৰ ফুটবল বিশ্বকাপ কোনে জিকিছিল?", "a5")


In [ ]:
# Inspect the saved state for a thread
graph.get_state({"configurable": {"thread_id": "t2"}}).values